In [9]:
# IMPORTS
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

DIR = "data/"
DATASET = DIR + "analysisofcountriesFCC.csv"

SEED = 42
TRAIN_SIZE = 0.7
TEST_SIZE = 1 - TRAIN_SIZE
OBJECTIVE = "number of sessions"

# 1.1. Carga y preparación del dataset

In [10]:
# Carga del dataset
df = pd.read_csv(DATASET, sep=";", na_values=["#N/A"])
df.head()

,country,number of sessions,internet population (descendent order),ses/intpop,ses/(48%*tot_population),connectivity_index,ordered by number of citygroups (1==0),ISO3136,engprof
0,Afghanistan,10,30,16,6,6,2,AF,35.0
1,Albania,28,29,48,63,77,2,AL,35.0
2,Algeria,51,67,28,17,12,4,DZ,40.0
3,Argentina,86,98,47,60,72,5,AR,60.0
4,Armenia,31,20,64,64,51,2,AM,35.0


In [11]:
display(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119 entries, 0 to 118
Data columns (total 9 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   country                                 119 non-null    object 
 1   number of sessions                      119 non-null    int64  
 2   internet population (descendent order)  119 non-null    int64  
 3   ses/intpop                              119 non-null    int64  
 4   ses/(48%*tot_population)                119 non-null    int64  
 5   connectivity_index                      119 non-null    int64  
 6   ordered by number of citygroups (1==0)  119 non-null    int64  
 7   ISO3136                                 119 non-null    object 
 8   engprof                                 115 non-null    float64
dtypes: float64(1), int64(6), object(2)
memory usage: 8.5+ KB


None

Al observar el dataset se observan las distintas variables:
- **country**: Variable categórica que contiene el nombre del país.
- **number of sessions**: Sesiones registradas a freeCodeCamp en el país.
- **internet population**: Población de dicho país con conexión a internet.
- **ses/intpop**: Relación entre internet population y number of sessions.
- **ses/(48%*tot_population)**: Relación entre internet population y number of sessions ajustada por un porcentaje.
- **connectivity_index**: Índice de conectividad del país.
- **ordered by number of citygroups (1==0)**: Indica la presencia de comunidades locales en el país.
- **ISO3136**: Variable categórica que contiene el código del país.
- **engprof**: Nivel del dominio del inglés en el país.

In [12]:
# Eliminación de nulos
display(df.isna().sum())
print("Número de filas antes de eliminar nulos:", len(df))
df = df.dropna()
print("Número de filas después de eliminar nulos:", len(df))

country                                   0
number of sessions                        0
internet population (descendent order)    0
ses/intpop                                0
ses/(48%*tot_population)                  0
connectivity_index                        0
ordered by number of citygroups (1==0)    0
ISO3136                                   0
engprof                                   4
dtype: int64

Número de filas antes de eliminar nulos: 119
Número de filas después de eliminar nulos: 115


Se observa que hay 4 registros con valores nulos en la variable **engprof**. Se van a eliminar dichos registros.

In [13]:
variables_numericas = df.select_dtypes(include="number")
rangos_intervalo = variables_numericas.agg(["min", "max"]).T
rangos_intervalo

,min,max
number of sessions,1.0,119.0
internet population (descendent order),2.0,119.0
ses/intpop,1.0,119.0
ses/(48%*tot_population),1.0,119.0
connectivity_index,1.0,119.0
ordered by number of citygroups (1==0),1.0,19.0
engprof,25.0,100.0


Al observar los datos se observa como no hay valores negativos en las variables numéricas, por lo que no se debe realizar tratamiento de valores negativos. 

Se va a observar si existen registros duplicados de la columna **country**

In [14]:
df["country"][df["country"].duplicated(keep=False)].sort_values()

Series([], Name: country, dtype: object)

Como se observa dicha variable no tiene ningún registro duplicado. En otras palabras, cada registro posee su propio valor. Por dicha razón la variable no es útil para las predicciones al no haber posibles patrones entre distintos registros. Por esto se va a eliminar la columna del dataset. Se va a realizar lo mismo con la columna "**ISO3136**", dado que esta representa el código de cada país. En otras palabras, representa la misma información.

In [15]:
df = df.drop(columns=["country","ISO3136"])

Por último también se va a eliminar **ses/intpop** y **ses/(48%*tot_population)** ya que sus valores dependen directamente de **number of sessions**, por lo que puede provocoar data leakage.

In [16]:
df = df.drop(columns=["ses/intpop","ses/(48%*tot_population)"])

In [17]:
df.duplicated().sum()

np.int64(0)

También se observa que no existen valores duplicados, por lo que no es requerido su tratamiento.

Al final se han realizado los siguientes pasos sobre el dataset:
- Eliminación de nulos.
- Eliminación de las variables **country**, **ISO3136**, **ses/intpop** y **ses/(48%*tot_population)**

El dataset acaba quedando de la siguiente manera:

In [18]:
df.head()

,number of sessions,internet population (descendent order),connectivity_index,ordered by number of citygroups (1==0),engprof
0,10,30,6,2,35.0
1,28,29,77,2,35.0
2,51,67,12,4,40.0
3,86,98,72,5,60.0
4,31,20,51,2,35.0


Por último se van a realizar los pasos finales:
- Estandarizar los valores.
- Dividir los datos en entrenamiento y validación.

La variable objetivo que se ha seleccionado es **number of sessions**. Esta variable representa el número de sesiones de freeCodeCamp asociadas a cada país, por lo que constituye el resultado principal que se quiere predecir. Y, dado que **number of sessions** es una variable numérica, el problema puede plantearse como una tarea de regresión. El objetivo del modelo es estimar cuántas sesiones puede generar un país a partir de variables presentes en el dataset.

In [19]:
X = df.drop(columns=[OBJECTIVE])
y = df[OBJECTIVE]

X_train, X_val, y_train, y_val = train_test_split(X,y,test_size=TEST_SIZE,random_state=SEED)

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

Por último se va a convertir de vuelta en dataframe

In [20]:
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_val_scaled = pd.DataFrame(X_val_scaled, columns=X_val.columns, index=X_val.index)

# 1.2. Entrenamiento del modelo

# 1.3. Explicación del modelo

# 1.4. Uso de IA generativa